# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedshereef1/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This notebook mirrors the deployed research paper section by section.
Every number in the paper traces back to a cell here.

> Skills loaded: `writing-research-papers` + `deploying-static-pages`

## 1. Question

**Research question:** Can a snapshot of search and engagement signals predict which content items are currently in decline — before traffic has visibly dropped?

**Decision it supports:** Content teams managing large portfolios need a ranked priority queue to decide which pages to refresh first. The alternative (a spreadsheet rule) is transparent but, as this work shows, performs *below chance* on held-out clients.

**Framing:** This is a classification task — predict `is_declining_label` (1 = declining, 0 = stable) — evaluated by Precision@K, which matches the editorial use case: of the top K items flagged, how many are actually declining?

## 2. Data

**Source:** FlyRank ML Internship starter dataset (`data/raw/content_refresh_anonymized.csv`)

**Size:** 30,000 rows × 44 columns, one row per pseudonymized content item

**Clients:** 32 pseudonymized client portfolios

**Time window:** Trailing 90-day snapshot (single cross-section — not a time series)

**Label:** `is_declining_label = 1` when `trend_direction == "down"`. `trend_direction` is derived from `trend_pct` (ratio of last-30 vs prior-30 impressions). Overall label rate: 54.2%.

**Excluded columns and why:**
- `trend_direction`, `trend_pct` — direct label source (leakage)
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` — overlap the label window (future leakage)
- `content_id`, `client_id` — identifiers only, not signals

**Key gotchas honoured:**
- Rate columns are ×100 percentages (0.76 = 0.76%, not 76%)
- `avg_position = 0` means no data — recoded to NaN, filled with median
- Missingness follows content_type — binary flags used instead of blind fillna(0)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Clients: {df['client_id'].nunique()}")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Overall label rate: {df['is_declining_label'].mean()*100:.1f}% declining")
print(f"Content types: {df['content_type'].value_counts().to_dict()}")

## 3. Methodology

**Feature set (29 features):**
- 21 numeric: log-transformed traffic (log_impressions_90d, log_clicks_90d, log_sessions_90d, log_ai_sessions_90d), engagement (ctr, scroll_rate, engagement_rate), position/freshness (avg_position, days_since_last_update, content_age_days), binary flags (has_clicks, has_ai_sessions, measurable_opportunity), and search signals (search_volume, competition, cpc, word_count, char_count, days_with_impressions, days_with_sessions, ai_traffic_pct)
- 8 categorical (label-encoded): competition_level, content_type, main_intent, age_tier, freshness_tier, word_count_tier, impression_tier, position_tier

**Models:**
1. Rule baseline — staleness ≥ Q75 (104d) → +2; search_volume ≥ Q75 (20) → +1
2. Logistic Regression — standard-scaled, C=1.0
3. Random Forest — 200 trees, max_depth=12, min_samples_leaf=10, seed=42

**Validation design:** Client-grouped 80/20 split — 26 train clients, 6 held-out test clients. Row-level random split was rejected because it allows client-level memorisation (inflates P@50 by +10pp).

**Leakage audit (7 checks):** All pass — see w06_validation_audit.ipynb. Deliberate injection of trend_pct pushed P@50 from 84% to 100%, confirming the harness detects real leakage.

**Primary metric:** Precision@K — matches the editorial use case (review top K items per week).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 42

# Feature engineering
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{col}"] = np.log1p(df[col].fillna(0))
df["has_clicks"]      = (df["clicks_90d"] > 0).astype(int)
df["has_ai_sessions"] = (df["ai_sessions_90d"] > 0).astype(int)
df["measurable_opportunity"] = ((df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)).astype(int)
df["avg_position"] = df["avg_position"].replace(0, np.nan).fillna(df["avg_position"].median())

NUMERIC_FEATURES = [
    "search_volume","competition","cpc","word_count","char_count",
    "log_impressions_90d","log_clicks_90d","log_sessions_90d","log_ai_sessions_90d",
    "days_with_impressions","days_with_sessions","content_age_days","days_since_last_update",
    "ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct",
    "has_clicks","has_ai_sessions","measurable_opportunity",
]
CATEGORICAL_FEATURES = [
    "competition_level","content_type","main_intent","age_tier",
    "freshness_tier","word_count_tier","impression_tier","position_tier",
]
for col in NUMERIC_FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
for col in CATEGORICAL_FEATURES:
    df[col] = df[col].fillna("unknown").astype(str)
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col])

ENC_FEATURES = [c + "_enc" for c in CATEGORICAL_FEATURES]
ALL_FEATURES = NUMERIC_FEATURES + ENC_FEATURES

# Client-grouped split
rng = np.random.default_rng(RANDOM_SEED)
clients = df["client_id"].unique().to_numpy()
rng.shuffle(clients)
n_test = max(1, int(len(clients) * 0.2))
test_clients  = set(clients[:n_test])
train_clients = set(clients[n_test:])

train_df = df[df["client_id"].isin(train_clients)].copy()
test_df  = df[df["client_id"].isin(test_clients)].copy()

X_train = train_df[ALL_FEATURES].values; y_train = train_df["is_declining_label"].values
X_test  = test_df[ALL_FEATURES].values;  y_test  = test_df["is_declining_label"].values

print(f"Train: {len(train_df):,} rows | {len(train_clients)} clients | label rate {y_train.mean()*100:.1f}%")
print(f"Test:  {len(test_df):,} rows  | {len(test_clients)} clients  | label rate {y_test.mean()*100:.1f}%")

## 4. Results (vs baseline)

All metrics on the same six held-out test clients. Base rate on test clients: 39.1%.

| Model | P@20 | P@50 | ROC-AUC | vs base rate |
|---|---|---|---|---|
| Base rate | 39.1% | 39.1% | — | 1.0× |
| Rule baseline (w04) | 30.0% | 28.0% | — | 0.7× (worse) |
| Logistic Regression | 25.0% | 36.0% | 0.706 | 0.9× |
| **Random Forest** | **90.0%** | **84.0%** | **0.760** | **2.1×** |

**Key finding:** The rule baseline performed *below the base rate* on held-out clients — a result that would not have been caught without a proper grouped validation split. The Random Forest improves Precision@50 by 3× over the rule and 2.1× over random chance.

In [ ]:
def precision_at_k(y_true, scores, k):
    idx = np.argsort(scores)[::-1][:k]
    return float(np.array(y_true)[idx].mean())

# Rule baseline
STALE_THRESH  = df["days_since_last_update"].quantile(0.75)
VOLUME_THRESH = df["search_volume"].quantile(0.75)
test_df["baseline_score"] = (
    (test_df["days_since_last_update"] >= STALE_THRESH).astype(int) * 2
    + (test_df["search_volume"].fillna(0) >= VOLUME_THRESH).astype(int)
)
base_p20 = precision_at_k(y_test, test_df["baseline_score"].values, 20)
base_p50 = precision_at_k(y_test, test_df["baseline_score"].values, 50)

# Logistic Regression
scaler = StandardScaler()
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED, C=1.0)
lr.fit(scaler.fit_transform(X_train), y_train)
lr_proba = lr.predict_proba(scaler.transform(X_test))[:, 1]
lr_p20 = precision_at_k(y_test, lr_proba, 20)
lr_p50 = precision_at_k(y_test, lr_proba, 50)
lr_auc = roc_auc_score(y_test, lr_proba)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_leaf=10,
                             random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_p20 = precision_at_k(y_test, rf_proba, 20)
rf_p50 = precision_at_k(y_test, rf_proba, 50)
rf_auc = roc_auc_score(y_test, rf_proba)
base_rate = y_test.mean()

results = pd.DataFrame([
    {"Model": "Base rate",          "P@20": f"{base_rate:.1%}", "P@50": f"{base_rate:.1%}", "ROC-AUC": "—",            "vs base rate": "1.0×"},
    {"Model": "Rule baseline (w04)","P@20": f"{base_p20:.1%}",  "P@50": f"{base_p50:.1%}",  "ROC-AUC": "—",            "vs base rate": f"{base_p50/base_rate:.1f}×"},
    {"Model": "Logistic Regression","P@20": f"{lr_p20:.1%}",   "P@50": f"{lr_p50:.1%}",   "ROC-AUC": f"{lr_auc:.3f}","vs base rate": f"{lr_p50/base_rate:.1f}×"},
    {"Model": "Random Forest",      "P@20": f"{rf_p20:.1%}",   "P@50": f"{rf_p50:.1%}",   "ROC-AUC": f"{rf_auc:.3f}","vs base rate": f"{rf_p50/base_rate:.1f}×"},
])
print("=" * 65)
print("ALL METRICS — same 6 held-out test clients")
print("=" * 65)
display(results.set_index("Model"))

## 5. Limitations

1. **Cross-sectional snapshot only** — one 90-day window per item; trajectory and seasonality are not modelled.
2. **Observational label, no causal claim** — patterns co-occur with decline; refreshing a flagged page is not guaranteed to reverse it.
3. **Six held-out clients = high variance** — 84% P@50 is measured on one draw of six clients from 32. A different draw could yield a different number.
4. **Feedly articles underserved** — ~100% missing keyword features; error rate 26.9% vs 41.6% for keyword articles on test set.
5. **No time-series component** — pages declining steeply and ones barely declining can look alike over 90-day aggregates.
6. **Boundary zone genuinely ambiguous** — 636 test rows (27.4%) have rf_proba 0.40–0.60; confident flags in this range misstate the evidence.

## 6. Ranked recommendations

From the action playbook (w07_action_playbook.ipynb) — all tiers are decision-support, not automation.

| Reason code | Threshold | Items | Action |
|---|---|---|---|
| `MODEL_HIGH_DECLINE` | rf_proba ≥ 0.70 | 8,369 (27.9%) | **Refresh content** |
| `MODEL_MOD_STALE` | 0.50–0.70 AND stale ≥ 104d | 3,451 (11.5%) | **Review freshness** |
| `MODEL_MOD_VOLUME` | 0.50–0.70 AND volume ≥ 20 | 2,362 (7.9%) | **Review opportunity** |
| `MONITOR` | rf_proba < 0.50 | 15,818 (52.7%) | **Monitor** |

**Must NOT automate:** content deletion, URL redirects, batch actioning at one threshold, client communications from raw scores.

**Retrain triggers:** P@50 < 65% on a fresh sample; label rate shifts ±10pp; new content type with >500 items; new 90-day snapshot (~quarterly).

## 7. Artifacts the paper embeds

Three figures are saved to `docs/figures/` and embedded in the deployed paper at `docs/index.html`:

- `fig1_model_vs_baseline.png` — bar chart, Precision@20 and @50 across all four models
- `fig2_reason_code_by_content_type.png` — action queue breakdown by content type
- `fig3_score_distribution.png` — histogram of rf_proba across all 30,000 items

These were generated by `w07_action_playbook.ipynb` and copied to `docs/figures/`.
The deployed paper lives at: `https://ahmedshereef1.github.io/flyrank-ml-internship/`

In [ ]:
# Verify the figure files exist for the paper
figures = [
    Path("../../docs/figures/fig1_model_vs_baseline.png"),
    Path("../../docs/figures/fig2_reason_code_by_content_type.png"),
    Path("../../docs/figures/fig3_score_distribution.png"),
]
print("Figure export verification:")
for p in figures:
    status = "OK" if p.exists() else "MISSING — run w07_action_playbook.ipynb first"
    print(f"  [{status}] {p.name}")

print(f"\nDeployed paper: https://ahmedshereef1.github.io/flyrank-ml-internship/")
print(f"Repo:           https://github.com/ahmedshereef1/flyrank-ml-internship")

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.